In [1]:
# No external AI API required
# Only built-in libraries are used
print("Environment Ready ✅")

Environment Ready ✅


In [20]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter API Key: ")

# Environment variable for API key (mock or real)
API_KEY = os.getenv("OPENAI_API_KEY")

if API_KEY is None:
    print("⚠ No API key found. Running in MOCK mode.")
else:
    print("API key loaded successfully.")

Enter API Key: ··········
API key loaded successfully.


In [2]:
import sqlite3
import json
from datetime import datetime

# Create database file
DB_FILE = "rayeva_ai.db"
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Create Products Table (Module 1)
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    primary_category TEXT,
    sub_category TEXT,
    seo_tags TEXT,
    sustainability_filters TEXT
)
""")

# Create Proposals Table (Module 2)
cursor.execute("""
CREATE TABLE IF NOT EXISTS proposals (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_type TEXT,
    budget REAL,
    total_cost REAL,
    proposal_json TEXT,
    impact_summary TEXT
)
""")

# Create AI Logs Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS ai_logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    module_name TEXT,
    prompt TEXT,
    response TEXT,
    timestamp TEXT
)
""")

conn.commit()
print("Database & Tables Created ✅")

Database & Tables Created ✅


In [3]:
def generate_product_category(product_name, description):

    # Simulated AI response
    result = {
        "primary_category": "Personal Care",
        "sub_category": "Oral Care",
        "seo_tags": ["eco", "bamboo", "biodegradable", "sustainable"],
        "sustainability_filters": ["plastic-free", "compostable"]
    }

    # Log AI interaction
    prompt = f"Generate category for {product_name}"

    cursor.execute("""
    INSERT INTO ai_logs (module_name, prompt, response, timestamp)
    VALUES (?, ?, ?, ?)
    """, ("Module 1", prompt, json.dumps(result), datetime.now().isoformat()))

    # Store product
    cursor.execute("""
    INSERT INTO products (name, primary_category, sub_category, seo_tags, sustainability_filters)
    VALUES (?, ?, ?, ?, ?)
    """, (
        product_name,
        result["primary_category"],
        result["sub_category"],
        json.dumps(result["seo_tags"]),
        json.dumps(result["sustainability_filters"])
    ))

    conn.commit()
    return result

print("Module 1 Ready ✅")

Module 1 Ready ✅


In [4]:
output1 = generate_product_category(
    "Bamboo Toothbrush",
    "Eco-friendly biodegradable toothbrush made from bamboo"
)

print(json.dumps(output1, indent=2))

{
  "primary_category": "Personal Care",
  "sub_category": "Oral Care",
  "seo_tags": [
    "eco",
    "bamboo",
    "biodegradable",
    "sustainable"
  ],
  "sustainability_filters": [
    "plastic-free",
    "compostable"
  ]
}


In [5]:
def generate_b2b_proposal(client_type, budget):

    recommended_products = [
        {"name": "Bamboo Toothbrush", "quantity": 100, "unit_cost": 50},
        {"name": "Steel Water Bottle", "quantity": 50, "unit_cost": 200}
    ]

    total_cost = sum(p["quantity"] * p["unit_cost"] for p in recommended_products)

    result = {
        "recommended_products": recommended_products,
        "total_estimated_cost": total_cost,
        "budget_remaining": budget - total_cost,
        "impact_summary": "Reduced plastic waste and promoted sustainable sourcing."
    }

    prompt = f"Generate proposal for {client_type} with budget {budget}"

    cursor.execute("""
    INSERT INTO ai_logs (module_name, prompt, response, timestamp)
    VALUES (?, ?, ?, ?)
    """, ("Module 2", prompt, json.dumps(result), datetime.now().isoformat()))

    cursor.execute("""
    INSERT INTO proposals (client_type, budget, total_cost, proposal_json, impact_summary)
    VALUES (?, ?, ?, ?, ?)
    """, (
        client_type,
        budget,
        total_cost,
        json.dumps(recommended_products),
        result["impact_summary"]
    ))

    conn.commit()
    return result

print("Module 2 Ready ✅")

Module 2 Ready ✅


In [6]:
output2 = generate_b2b_proposal("Corporate Gifting", 50000)

print(json.dumps(output2, indent=2))

{
  "recommended_products": [
    {
      "name": "Bamboo Toothbrush",
      "quantity": 100,
      "unit_cost": 50
    },
    {
      "name": "Steel Water Bottle",
      "quantity": 50,
      "unit_cost": 200
    }
  ],
  "total_estimated_cost": 15000,
  "budget_remaining": 35000,
  "impact_summary": "Reduced plastic waste and promoted sustainable sourcing."
}


In [7]:
cursor.execute("SELECT * FROM ai_logs ORDER BY id DESC")
logs = cursor.fetchall()

for log in logs:
    print("\nLog Entry:")
    print(log)


Log Entry:
(2, 'Module 2', 'Generate proposal for Corporate Gifting with budget 50000', '{"recommended_products": [{"name": "Bamboo Toothbrush", "quantity": 100, "unit_cost": 50}, {"name": "Steel Water Bottle", "quantity": 50, "unit_cost": 200}], "total_estimated_cost": 15000, "budget_remaining": 35000, "impact_summary": "Reduced plastic waste and promoted sustainable sourcing."}', '2026-02-26T09:13:41.026542')

Log Entry:
(1, 'Module 1', 'Generate category for Bamboo Toothbrush', '{"primary_category": "Personal Care", "sub_category": "Oral Care", "seo_tags": ["eco", "bamboo", "biodegradable", "sustainable"], "sustainability_filters": ["plastic-free", "compostable"]}', '2026-02-26T09:13:06.536998')


In [8]:
from google.colab import files
files.download("rayeva_ai.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
cursor.execute("SELECT * FROM products")
print(cursor.fetchall())

[(1, 'Bamboo Toothbrush', 'Personal Care', 'Oral Care', '["eco", "bamboo", "biodegradable", "sustainable"]', '["plastic-free", "compostable"]')]


In [10]:
cursor.execute("SELECT * FROM proposals")
print(cursor.fetchall())

[(1, 'Corporate Gifting', 50000.0, 15000.0, '[{"name": "Bamboo Toothbrush", "quantity": 100, "unit_cost": 50}, {"name": "Steel Water Bottle", "quantity": 50, "unit_cost": 200}]', 'Reduced plastic waste and promoted sustainable sourcing.')]


In [11]:
generate_product_category("Organic Cotton Bag", "Reusable eco shopping bag")
generate_product_category("Solar Power Bank", "Portable solar charging device")

generate_b2b_proposal("Startup Welcome Kit", 30000)
generate_b2b_proposal("College Eco Campaign", 20000)

{'recommended_products': [{'name': 'Bamboo Toothbrush',
   'quantity': 100,
   'unit_cost': 50},
  {'name': 'Steel Water Bottle', 'quantity': 50, 'unit_cost': 200}],
 'total_estimated_cost': 15000,
 'budget_remaining': 5000,
 'impact_summary': 'Reduced plastic waste and promoted sustainable sourcing.'}

In [15]:
def ai_generate_product_category(product_name, description):
    """
    AI Layer - Responsible ONLY for generating structured output
    """
    return {
        "primary_category": "Personal Care",
        "sub_category": "Oral Care",
        "seo_tags": ["eco", "bamboo", "biodegradable"],
        "sustainability_filters": ["plastic-free"]
    }

In [16]:
def process_product(product_name, description):
    """
    Business Logic Layer
    - Calls AI
    - Validates output
    - Stores in DB
    """

    result = ai_generate_product_category(product_name, description)

    # Validation
    required_keys = ["primary_category", "sub_category", "seo_tags", "sustainability_filters"]
    for key in required_keys:
        if key not in result:
            raise ValueError(f"Missing key: {key}")

    # Store in database
    cursor.execute("""
        INSERT INTO products (name, primary_category, sub_category, seo_tags, sustainability_filters)
        VALUES (?, ?, ?, ?, ?)
    """, (
        product_name,
        result["primary_category"],
        result["sub_category"],
        json.dumps(result["seo_tags"]),
        json.dumps(result["sustainability_filters"])
    ))

    conn.commit()

    return result

In [17]:
def log_ai_interaction(module_name, prompt, response):
    cursor.execute("""
        INSERT INTO ai_logs (module_name, prompt, response, timestamp)
        VALUES (?, ?, ?, ?)
    """, (module_name, prompt, json.dumps(response), datetime.now().isoformat()))
    conn.commit()

In [18]:
try:
    output = process_product("Bamboo Toothbrush", "Eco product")
    print(json.dumps(output, indent=2))
except Exception as e:
    print("Error occurred:", str(e))

{
  "primary_category": "Personal Care",
  "sub_category": "Oral Care",
  "seo_tags": [
    "eco",
    "bamboo",
    "biodegradable"
  ],
  "sustainability_filters": [
    "plastic-free"
  ]
}


In [12]:
print("Rayeva AI System Running Successfully ✅")

Rayeva AI System Running Successfully ✅
